# Vectora — run 3b (the GPU-only remainder of run 3)

Uses the **new defaults** (BGE-small, MMR off, floors calibrated on 50 negatives) everywhere except G's old-defaults arm. Run 3 (Colab, 2026-10-20) finished retrieval (A), calibration (B) and fast-mode answers for qwen2.5:7b (C), then ran out of GPU. The ablation moved to the laptop CPU (`vectora ablate --no-llm`). This notebook runs only what needs a GPU:

| # | Experiment | GPU time |
|---|---|---|
| C2 | fast-mode answers with **gemma3:4b** (died at 244/373 last time) | ~25 min |
| D | **deep** mode, first 150 questions + negatives: grounding judge on, judged by qwen2.5:7b | ~60–90 min |
| F | export 30 judged answers for hand labelling | seconds |
| G | balanced **with LLM**: old defaults (MiniLM, MMR, −4.0) vs new (BGE-small, no MMR, 0.903) | ~80 min |

Plus ~15 min of setup (Ollama, models, corpus, index). About 3–3.5 hours in total.

**Before you leave it:** Runtime → Change runtime type → **T4 GPU**. Run **cell 1 alone** and approve Google Drive.
Check cell 2 prints a GPU line (not "no compatible GPU"). Then Runtime → **Run all**, and leave the tab open with the laptop awake and plugged in.

Each experiment saves to `MyDrive/vectora_results/<run>/` as it finishes, so a disconnect loses at most the one in progress. 

## 1. Connect Drive (needs you awake)

In [ ]:
import datetime, os
RUN = "run3b_" + datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/vectora_results/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. Ollama on the GPU

In [ ]:
ANSWER = "gemma3:4b"     # run 2's best answer model
JUDGE  = "qwen2.5:7b"    # utility + judge: a different model, so it does not grade itself
MODELS = [ANSWER, JUDGE]

# pciutils + lshw let the installer detect the GPU; without them it installs a CPU-only build.
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": "4",
                               "OLLAMA_MAX_LOADED_MODELS": "2", "OLLAMA_CONTEXT_LENGTH": "8192"})
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); print("ollama up"); break
    except Exception:
        time.sleep(1)
for m in MODELS:
    !ollama pull {m} 2>&1 | tail -1
!ollama run {ANSWER} "hi" > /dev/null 2>&1; ollama ps
!grep -i -E "inference compute|no compatible gpu" /content/ollama.log | head -3

## 3. Vectora + config

In [ ]:
!git clone -q https://github.com/SiddharthaRaja/Vectora.git /content/Vectora
%cd /content/Vectora
!git log --oneline -1 | tee {OUT}/commit.txt
!pip install -q -e ".[ingest,eval]" 2>&1 | grep -v -i "warning\|dependency resolver\|incompatible" | tail -2
open(".env", "w").write(f"""
OPENAI_API_KEY=ollama
OPENAI_BASE_URL=http://127.0.0.1:11434/v1
VECTORA_ANSWER_MODEL={ANSWER}
VECTORA_UTILITY_MODEL={JUDGE}
VECTORA_DAILY_COST_CEILING_USD=0
VECTORA_LLM_TIMEOUT_SECONDS=180
VECTORA_LLM_MAX_RETRIES=2
VECTORA_CORPUS=qasper-300
VECTORA_QDRANT_URL=
VECTORA_QDRANT_PATH=./data/qdrant
VECTORA_CORS_ORIGINS=http://localhost:5000
VECTORA_LOG_LEVEL=WARNING
""")

## 4. Build the 300-paper corpus, ingest, index

Same command as on your laptop, so the same 300 papers and 365 title-free questions (the golden set is also in git; `--force` rebuilds it identically).
Every case has the annotators' evidence paragraphs and all their answers.

In [ ]:
!python scripts/qasper_corpus.py --corpus qasper-300 --split all --papers 300 --force 2>&1 | tail -6
!git diff --stat corpora/qasper-300/golden.yaml | tail -1 | tee {OUT}/golden_diff.txt   # empty = identical to the committed set
!date; vectora ingest 2>&1 | tail -3
!vectora index --recreate 2>&1 | tail -3; date
!cp corpora/qasper-300/golden.yaml {OUT}/golden.yaml

## C2. gemma3:4b answers in `fast` mode

New in the report: **raw citation validity / hallucinated markers** (what the model wrote, before Vectora strips bad citations) and **answer token F1 / token recall** against QASPER's gold answers.
Mechanical scores only (`--no-judge`), so this is quick.

In [ ]:
!date
for m in [ANSWER]:   # qwen2.5:7b already done in run 3
    tag = m.replace(":", "-")
    print("=====", m)
    !VECTORA_ANSWER_MODEL={m} vectora answers --mode fast --no-judge --output {OUT}/expC_fast_{tag}.json 2>&1 | tee {OUT}/expC_fast_{tag}.log | grep -A22 -i "mechanical" | head -24

## 8. Experiment D: `deep` mode with the faithfulness judge

Deep mode runs the grounding check (judged by `qwen2.5:7b`) and the relevance judge. First 150 questions (+ the negatives the harness keeps), roughly 60–90 minutes.

In [ ]:
!date
!vectora answers --mode deep --limit 150 --output {OUT}/expD_deep.json 2>&1 | tee {OUT}/expD_deep.log | tail -45
!date

## 10. Experiment F: export 30 answers for hand labelling

Writes `judge_labels.csv` to your Drive folder. Later, on your laptop: open it, **hide the `judge_*` columns**, and for each row put `1` in `human_faithful` if every claim in the answer is supported by the sources shown, `0` if any claim is not.
Then send it to me and I'll compute agreement.

In [ ]:
!python -m evaluation.judge_agreement export --answers {OUT}/expD_deep.json --chunks corpora/qasper-300/chunks.jsonl --out {OUT}/judge_labels.csv --n 30

## G. Old defaults vs new defaults, with the LLM (balanced)

Old (run 3): all-MiniLM-L6-v2, MMR on, rerank floor −4.0. New (2026-10-21): bge-small-en-v1.5, MMR off, rerank floor 0.903 — chosen from CPU experiments without the LLM. This confirms them with query expansion on. The MiniLM index is a second corpus built from the **same chunks**. ~80 minutes.

In [ ]:
!date
import shutil, pathlib, os, json, random, subprocess
OLD = {"VECTORA_EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2", "VECTORA_EMBEDDING_DIM": "384",
       "VECTORA_USE_MMR": "true", "VECTORA_MIN_RERANK_SCORE": "-4.0", "VECTORA_MIN_DENSE_SCORE": "0.40"}
env_old = {**os.environ, **OLD}
subprocess.run('vectora corpus create q300-minilm -d "qasper-300, MiniLM (old default)" --kind "NLP research papers"', shell=True, env=env_old)
for f in ("chunks.jsonl", "golden.yaml"):
    shutil.copy(pathlib.Path("corpora/qasper-300") / f, pathlib.Path("corpora/q300-minilm") / f)
subprocess.run("vectora --corpus q300-minilm index --recreate 2>&1 | tail -2", shell=True, env=env_old)
subprocess.run(f"vectora --corpus q300-minilm eval --mode balanced --output {OUT}/expG_balanced_old.json 2>&1 | tee {OUT}/expG_balanced_old.log | grep -A6 'mode  '", shell=True, env=env_old)
!vectora eval --mode balanced --output {OUT}/expG_balanced_new.json 2>&1 | tee {OUT}/expG_balanced_new.log | grep -A6 "mode  "
a = json.load(open(f"{OUT}/expG_balanced_old.json"))["reports"][0]
b = json.load(open(f"{OUT}/expG_balanced_new.json"))["reports"][0]
fa, fb, n = set(a["misses"]), set(b["misses"]), a["cases"]
fixed, broken = len(fa - fb), len(fb - fa)
diff = [1] * fixed + [-1] * broken + [0] * (n - fixed - broken)
random.seed(0); boots = sorted(sum(random.choice(diff) for _ in range(n)) / n for _ in range(4000))
print(f"passage recall {a['recall']:.3f} -> {b['recall']:.3f}  (fixed {fixed}, broke {broken}, 95% CI {boots[100]:+.3f} to {boots[3899]:+.3f})")
print(f"paper recall {a['paper_recall']:.3f} -> {b['paper_recall']:.3f};  off-corpus refused {a['negative_rejection_rate']:.0%} -> {b['negative_rejection_rate']:.0%};  p50 {a['median_latency_ms']:.0f} -> {b['median_latency_ms']:.0f} ms")
!date

## 11. Wrap up

In [ ]:
!cp corpora/qasper-300/reports/* {OUT}/ 2>/dev/null; cp /content/ollama.log {OUT}/ 2>/dev/null; date | tee {OUT}/finished_at.txt
!ls -la {OUT}